# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [48]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE



In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [9]:
fraud_count = fraud.sum()
display(fraud_count)

distance_from_home                2.662879e+07
distance_from_last_transaction    5.036519e+06
ratio_to_median_purchase_price    1.824182e+06
repeat_retailer                   8.815360e+05
used_chip                         3.503990e+05
used_pin_number                   1.006080e+05
online_order                      6.505520e+05
fraud                             8.740300e+04
dtype: float64

In [18]:
fraud_col = fraud['fraud'].value_counts()
fraud_col

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [22]:
fraud_col_p = fraud['fraud'].value_counts(normalize=True)
fraud_percentages = (fraud_col_p * 100).round(2)
print(f"In percentage: {fraud_percentages}")


In percentage: fraud
0.0    91.26
1.0     8.74
Name: proportion, dtype: float64


1. What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

        It is an imbalanced set because the fraud cases are the 91.26% while only 8.74% are not fraud.

In [24]:
features = fraud.drop(columns =['fraud'])
target = fraud['fraud']
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [38]:
scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

display(X_train_scaled.head(5))
display(X_test_scaled.head(5))

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order
0,0.000331,0.000083,0.002352,1.0,1.0,0.0,1.0
1,0.003688,0.000002,0.002246,1.0,1.0,0.0,1.0
2,0.005209,0.000082,0.001718,1.0,0.0,0.0,1.0
3,0.000056,0.000045,0.001963,0.0,1.0,0.0,0.0
4,0.002078,0.000002,0.002453,1.0,1.0,0.0,0.0


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order
0,0.000344,0.000009,0.016056,1.0,0.0,0.0,1.0
1,0.000074,0.000131,0.002727,0.0,0.0,0.0,1.0
2,0.000136,0.000089,0.002303,0.0,0.0,0.0,1.0
3,0.000124,0.000039,0.001571,0.0,1.0,0.0,0.0
4,0.000915,0.001050,0.015838,1.0,1.0,0.0,1.0


In [39]:
scaler = StandardScaler()
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


- **2.** Train a LogisticRegression.

In [27]:
scaled_train = scaler.transform(X_train)
scaled_test = scaler.transform(X_test)

X_train_standarized_df = pd.DataFrame(scaled_train, columns=X_train.columns, index=X_train.index)
X_test_standarized_df = pd.DataFrame(scaled_test, columns=X_test.columns, index=X_test.index)

In [30]:
log_reg = LogisticRegression()
log_reg.fit(X_train_standarized_df, y_train)
y_pred_test = log_reg.predict(X_test_standarized_df)

**3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

In [35]:
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182615
         1.0       0.89      0.60      0.72     17385

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



In [36]:
print("Accuracy", log_reg.score(X_test_standarized_df, y_test))
print("Precision", precision_score(y_test, y_pred_test)) 
print("Recall", recall_score(y_test, y_pred_test)) 

Accuracy 0.95922
Precision 0.892156029574233
Recall 0.6038538970376761


Precission is important because the dataset is strongly imbalanced


Recall is also important here when the dataset is that imbalance, it has problems with low amount statistics

**4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 

In [50]:
from imblearn.over_sampling import RandomOverSampler

over_sampling = RandomOverSampler (random_state = 0)
X_train_over, y_train_over = over_sampling.fit_resample (X_train_scaled,y_train)

log_reg_overs = LogisticRegression(max_iter=1000)
log_reg_overs.fit(X_train_over, y_train_over)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [51]:
y_pred_test_os = log_reg_overs.predict(X_test_scaled)
print(classification_report(y_pred = y_pred_test_os, y_true = y_test))

              precision    recall  f1-score   support

         0.0       0.99      0.92      0.96    182615
         1.0       0.54      0.93      0.68     17385

    accuracy                           0.92    200000
   macro avg       0.76      0.93      0.82    200000
weighted avg       0.95      0.92      0.93    200000



 **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [49]:
under_sampling = RandomUnderSampler(random_state=0)
X_train_under, y_train_under = under_sampling.fit_resample(X_train_scaled,y_train)

log_reg_unders = LogisticRegression(max_iter=1000)
log_reg_unders.fit(X_train_under, y_train_under)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [52]:
y_pred_test_os = log_reg_unders.predict(X_test_scaled)
print(classification_report(y_pred = y_pred_test_os, y_true = y_test))

              precision    recall  f1-score   support

         0.0       0.99      0.89      0.94    182615
         1.0       0.45      0.91      0.60     17385

    accuracy                           0.90    200000
   macro avg       0.72      0.90      0.77    200000
weighted avg       0.94      0.90      0.91    200000



- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [46]:
sm = SMOTE(random_state = 1,sampling_strategy=1.0)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled,y_train)

log_reg_smote = LogisticRegression(max_iter=1000)
log_reg_smote.fit(X_train_sm, y_train_sm)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [47]:
y_pred_test_smote = log_reg_smote.predict(X_test_scaled)
print(classification_report(y_pred = y_pred_test_smote, y_true = y_test))

              precision    recall  f1-score   support

         0.0       0.99      0.92      0.96    182615
         1.0       0.54      0.93      0.68     17385

    accuracy                           0.92    200000
   macro avg       0.77      0.93      0.82    200000
weighted avg       0.95      0.92      0.93    200000

